In [1]:
import pandas as pd
import requests
import os

# Topic 1: Linear Regression

The discussion will potentially cover the following questions:

* How to select explanatory variables?
* How to choose the right model and why? 
* How to interpret model results? How to interpret regression coefficients, such as margin, importance?

Specifically for predictive model, how to evaluate the predictive power of the model? e.g. in-sample v.s out-of-sample model errors, type I and type II errors, etc.

## Load Data

In [2]:
# Define get_example_ds function to download dataset to the current directory
def get_example_ds(stata_url,local_filename):
    try:
        # Download the file
        response = requests.get(stata_url, timeout=10)
        response.raise_for_status()  # Raise error for bad status codes

        # Save locally
        with open(local_filename, "wb") as f:
            f.write(response.content)
        print(f"Downloaded dataset to {os.path.abspath(local_filename)}")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading dataset: {e}")
    except (ValueError, OSError) as e:
        print(f"Error reading dataset: {e}")

In [3]:
# Example: Download Stata's "auto.dta" dataset - Car
stata_url = "https://www.stata-press.com/data/r18/auto.dta"  # Change r18 to your Stata release if needed
local_filename = "auto.dta"

# Download dataset
get_example_ds(stata_url,local_filename)

# Load into pandas DataFrame
df_auto = pd.read_stata(local_filename)

# Preview the dataframe
df_auto.head()

Downloaded dataset to C:\Users\yuan.yuan\OneDrive - Summit, LLC\Documents\Python\auto.dta


,make,price,mpg,rep78,headroom,trunk,weight,length,turn,displacement,gear_ratio,foreign
0,AMC Concord,4099,22,3.0,2.5,11,2930,186,40,121,3.58,Domestic
1,AMC Pacer,4749,17,3.0,3.0,11,3350,173,40,258,2.53,Domestic
2,AMC Spirit,3799,22,NaN,3.0,12,2640,168,35,121,3.08,Domestic
3,Buick Century,4816,20,3.0,4.5,16,3250,196,40,196,2.93,Domestic
4,Buick Electra,7827,15,4.0,4.0,20,4080,222,43,350,2.41,Domestic


## Modeling - Linear Regression

In the first modeling exercise, we want to predict the gas efficiency of cars using car characteristics and potentially market variables.

In the car dataset, the `mpg` variable refers to miles per gallon. This dataset also provide other features such as the price, make, weight, mechanic and producer related informations. 

### OLS Regression

In [4]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

In [5]:
df_auto=pd.read_stata(r'C:\Users\yuan.yuan\OneDrive - Summit, LLC\Documents\Python\auto.dta')

* make: The make and model of the car (string).
* price: The price of the car (integer).
* mpg: Mileage in miles per gallon (integer).
* rep78: Repair record for 1978 (integer).
* headroom: Headroom in inches (float).
* trunk: Trunk space in cubic feet (integer).
* weight: Weight of the car in pounds (integer).
* length: Length of the car in inches (integer).
* turn: Turn circle in feet (integer).
* displacement: Engine displacement in cubic inches (integer).
* gear_ratio: Gear ratio (float).
* foreign: Indicates whether the car is domestic or foreign (byte).

In [54]:
df_auto['make'].unique()

array(['AMC Concord', 'AMC Pacer', 'AMC Spirit', 'Buick Century',
       'Buick Electra', 'Buick LeSabre', 'Buick Opel', 'Buick Regal',
       'Buick Riviera', 'Buick Skylark', 'Cad. Deville', 'Cad. Eldorado',
       'Cad. Seville', 'Chev. Chevette', 'Chev. Impala', 'Chev. Malibu',
       'Chev. Monte Carlo', 'Chev. Monza', 'Chev. Nova', 'Dodge Colt',
       'Dodge Diplomat', 'Dodge Magnum', 'Dodge St. Regis', 'Ford Fiesta',
       'Ford Mustang', 'Linc. Continental', 'Linc. Mark V',
       'Linc. Versailles', 'Merc. Bobcat', 'Merc. Cougar',
       'Merc. Marquis', 'Merc. Monarch', 'Merc. XR-7', 'Merc. Zephyr',
       'Olds 98', 'Olds Cutl Supr', 'Olds Cutlass', 'Olds Delta 88',
       'Olds Omega', 'Olds Starfire', 'Olds Toronado', 'Plym. Arrow',
       'Plym. Champ', 'Plym. Horizon', 'Plym. Sapporo', 'Plym. Volare',
       'Pont. Catalina', 'Pont. Firebird', 'Pont. Grand Prix',
       'Pont. Le Mans', 'Pont. Phoenix', 'Pont. Sunbird', 'Audi 5000',
       'Audi Fox', 'BMW 320i', 'Dats

In [53]:
# Can we use the dataset directly? 
col_y = 'mpg'
list_col_x = ['make', 'price', 'rep78', 'headroom', 'trunk', 'weight','length', 'turn', 'displacement', 'gear_ratio', 'foreign']
y = df_auto[col_y]
X = df_auto[list_col_x]

# Try the regular OLS method
model = sm.OLS(y, X).fit()
print(model.summary())

ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

**Let's prepare the input dataset X properly**

In [55]:
# Examining the data
df_auto.head()
df_auto.describe()

,price,mpg,rep78,headroom,trunk,weight,length,turn,displacement,gear_ratio
count,74.000000,74.000000,69.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000
mean,6165.256757,21.297297,3.405797,2.993243,13.756757,3019.459459,187.932432,39.648649,197.297297,3.014865
std,2949.495885,5.785503,0.989932,0.845995,4.277404,777.193567,22.266340,4.399354,91.837219,0.456287
min,3291.000000,12.000000,1.000000,1.500000,5.000000,1760.000000,142.000000,31.000000,79.000000,2.190000
25%,4220.250000,18.000000,3.000000,2.500000,10.250000,2250.000000,170.000000,36.000000,119.000000,2.730000
50%,5006.500000,20.000000,3.000000,3.000000,14.000000,3190.000000,192.500000,40.000000,196.000000,2.955000
75%,6332.250000,24.750000,4.000000,3.500000,16.750000,3600.000000,203.750000,43.000000,245.250000,3.352500
max,15906.000000,41.000000,5.000000,5.000000,23.000000,4840.000000,233.000000,51.000000,425.000000,3.890000


In [7]:
# Missing data imputation, e.g rep78, repair record in 1978, 1=poor - 5=excellent
df_data=df_auto.copy()
#df_data=df_data.dropna() # can simply drop missing data
df_data['rep_missing']=np.where(df_data['rep78'].isna(),1,0)
df_data.loc[df_data['rep78'].isna(),'rep78']=0
df_data.columns

Index(['make', 'price', 'mpg', 'rep78', 'headroom', 'trunk', 'weight',
       'length', 'turn', 'displacement', 'gear_ratio', 'foreign',
       'rep_missing'],
      dtype='object')

In [9]:
# Naive Way: Generate dummies for categorical variables
col_y = 'mpg'
y = df_data[col_y]

list_col_cat=['make','foreign','rep_missing']
list_col_cont=['rep78', 'headroom', 'trunk', 'weight',
       'length', 'turn', 'displacement', 'gear_ratio',]
X =pd.get_dummies(df_data[list_col_cat+list_col_cont], columns=list_col_cat, drop_first=True)
X = sm.add_constant(X)  # add intercept

# OLS
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Thu, 25 Jun 2026   Prob (F-statistic):                nan
Time:                        11:01:33   Log-Likelihood:                 1811.2
No. Observations:                  74   AIC:                            -3474.
Df Residuals:                       0   BIC:                            -3304.
Df Model:                          73                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      6

C:\Users\yuan.yuan\Anaconda3\lib\site-packages\statsmodels\regression\linear_model.py:1794: RuntimeWarning: divide by zero encountered in true_divide
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
C:\Users\yuan.yuan\Anaconda3\lib\site-packages\statsmodels\regression\linear_model.py:1794: RuntimeWarning: invalid value encountered in double_scalars
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
C:\Users\yuan.yuan\Anaconda3\lib\site-packages\statsmodels\regression\linear_model.py:1716: RuntimeWarning: divide by zero encountered in double_scalars
  return np.dot(wresid, wresid) / self.df_resid


In [56]:
# Re-do the categorical

df_data['make_r']=df_data['make'].apply(lambda x: x.split(' ')[0])
print(df_data['make_r'].unique())

col_y = 'mpg'
y = df_data[col_y]

list_col_cat=['make_r','foreign']
list_col_cont=['rep78', 'headroom', 'trunk', 'weight',
       'length', 'turn', 'displacement', 'gear_ratio', 'rep_missing']
X =pd.get_dummies(df_data[list_col_cat+list_col_cont], columns=list_col_cat, drop_first=True)
X = sm.add_constant(X)  # add intercept

# Try the regular OLS method
model = sm.OLS(y, X).fit()
print(model.summary())

['AMC' 'Buick' 'Cad.' 'Chev.' 'Dodge' 'Ford' 'Linc.' 'Merc.' 'Olds'
 'Plym.' 'Pont.' 'Audi' 'BMW' 'Datsun' 'Fiat' 'Honda' 'Mazda' 'Peugeot'
 'Renault' 'Subaru' 'Toyota' 'VW' 'Volvo']
                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.819
Model:                            OLS   Adj. R-squared:                  0.686
Method:                 Least Squares   F-statistic:                     6.133
Date:                Thu, 25 Jun 2026   Prob (F-statistic):           5.97e-08
Time:                        12:22:04   Log-Likelihood:                -171.14
No. Observations:                  74   AIC:                             406.3
Df Residuals:                      42   BIC:                             480.0
Df Model:                          31                                         
Covariance Type:            nonrobust                                         
                      coef 

In [11]:
# Try fit a R style equation, customize the equation by yourself
equation = 'mpg ~ rep78:rep_missing + weight + displacement + C(foreign)'
model_r = smf.ols(equation, data=df_data).fit()
print(model_r.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.663
Model:                            OLS   Adj. R-squared:                  0.648
Method:                 Least Squares   F-statistic:                     45.88
Date:                Thu, 25 Jun 2026   Prob (F-statistic):           1.64e-16
Time:                        11:01:51   Log-Likelihood:                -194.16
No. Observations:                  74   AIC:                             396.3
Df Residuals:                      70   BIC:                             405.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                41.84

### How to evaluate the model fit? 

* Statistics

    * R² and Adj. R²: Measures the proportion of variance in the dependent variable explained by your model. (Closer to 1.0 is better).
    * Log-Likelihood: Measures the probability of observing the given data under your fitted model. Useful for comparing two models (higher values are better).
    * AIC & BIC: Penalized versions of log-likelihood that account for the number of predictors. These are useful for comparing multiple models (lower values are better).

* Hypothesis Testing

    * F-statistic: Tests the overall significance of the model. If the Prob(F-statistic) is lower than your significance level (e.g., 0.05), the model fits significantly better than a flat line with no predictors.
    * p-values: Located in the coefficient table, these evaluate if individual predictors meaningfully contribute to the model.
    
* Evaluate Underline Assumptions

    * Residual Normality: Check the Omnibus test or Jarque-Bera (JB) test in the summary (low p-values indicate residuals are not normally distributed).
    * Autocorrelation: Use the Durbin-Watson statistic. A value near 2.0 suggests no autocorrelation in the residuals.
    * Multicollinearity: Check the Condition Number. Values above 20 to 30 often indicate severe multicollinearity.

In [12]:
print(model.summary())
print('AIC: ', model.aic)
print('BIC: ', model.bic)

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.819
Model:                            OLS   Adj. R-squared:                  0.686
Method:                 Least Squares   F-statistic:                     6.133
Date:                Thu, 25 Jun 2026   Prob (F-statistic):           5.97e-08
Time:                        11:01:59   Log-Likelihood:                -171.14
No. Observations:                  74   AIC:                             406.3
Df Residuals:                      42   BIC:                             480.0
Df Model:                          31                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              44.3788     11.040     

In [13]:
# Evaluate Model Performance, in-sample

from statsmodels.tools.eval_measures import mse

# Generate predictions from your fitted model
predictions = model.predict(X)

# Calculate the Mean Squared Error
error = mse(y, predictions)
print(f"Mean Squared Error: {error}")

Mean Squared Error: 5.974276240836366


### Logistic Regression

* Convert the `mpg` variable to a categorical variable as follows and fit a logistic regression model. 
* How should we interprete the model result?
* How to evaluate the model performance?

In [65]:
# Generate categorical variable mpg_bins
labels=['low','high']
cutoff=25
bins=[1,cutoff,df_auto['mpg'].max()+1]
df_data['mpg_bins'] = pd.cut(x=df_data['mpg'], bins=bins,labels=labels)
df_data.head()
df_data.pivot_table(index='mpg_bins',aggfunc='count')

,displacement,foreign,gear_ratio,headroom,length,make,make_r,mpg,price,rep78,rep_missing,trunk,turn,weight
mpg_bins,,,,,,,,,,,,,,
low,60,60,60,60,60,60,60,60,60,60,60,60,60,60
high,14,14,14,14,14,14,14,14,14,14,14,14,14,14


In [66]:
df_data['mpg_bins']=np.where(df_data['mpg']>=cutoff, 1, 0)

In [67]:
# Customize the equation and fit the logistic regression model
equation = 'mpg_bins ~ length + weight + displacement + C(foreign)'
model = smf.logit(formula=equation, data=df_data)
results = model.fit()
print(results.summary())

Optimization terminated successfully.
         Current function value: 0.209042
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:               mpg_bins   No. Observations:                   74
Model:                          Logit   Df Residuals:                       69
Method:                           MLE   Df Model:                            4
Date:                Thu, 25 Jun 2026   Pseudo R-squ.:                  0.6330
Time:                        12:47:29   Log-Likelihood:                -15.469
converged:                       True   LL-Null:                       -42.153
Covariance Type:            nonrobust   LLR p-value:                 7.137e-11
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                32.4035     11.069      2.927      0.003      10.708      54.

**Confusion Matrix**

* matrix[0, 0]: True Negatives (TN)
* matrix[0, 1]: False Positives (FP)
* matrix[1, 0]: False Negatives (FN)
* matrix[1, 1]: True Positives (TP)

In [70]:
confusion_matrix = results.pred_table(threshold=0.3)
print(confusion_matrix)

[[48.  7.]
 [ 3. 16.]]


In [71]:
# Plot ROC AUC

## Rank the variable importance

In [20]:
df_data.corr()

,price,mpg,rep78,headroom,trunk,weight,length,turn,displacement,gear_ratio,rep_missing,mpg_bins
price,1.000000,-0.468597,-0.011667,0.114506,0.314332,0.538611,0.431831,0.309617,0.494943,-0.313661,0.024364,-0.253145
mpg,-0.468597,1.000000,0.289326,-0.413803,-0.581585,-0.807175,-0.795779,-0.719186,-0.705643,0.616177,0.004811,0.793249
rep78,-0.011667,0.289326,1.000000,-0.087037,-0.014539,-0.252377,-0.223958,-0.274666,-0.280459,0.212842,-0.669314,0.258238
headroom,0.114506,-0.413803,-0.087037,1.000000,0.662011,0.483456,0.516295,0.424465,0.474491,-0.377852,-0.029872,-0.289796
trunk,0.314332,-0.581585,-0.014539,0.662011,1.000000,0.672206,0.726596,0.601059,0.608635,-0.508665,-0.149331,-0.410515
weight,0.538611,-0.807175,-0.252377,0.483456,0.672206,1.000000,0.946009,0.857443,0.894896,-0.759258,-0.060490,-0.666827
length,0.431831,-0.795779,-0.223958,0.516295,0.726596,0.946009,1.000000,0.864261,0.835140,-0.696383,-0.060038,-0.698990
turn,0.309617,-0.719186,-0.274666,0.424465,0.601059,0.857443,0.864261,1.000000,0.776765,-0.676300,-0.126210,-0.632379
displacement,0.494943,-0.705643,-0.280459,0.474491,0.608635,0.894896,0.835140,0.776765,1.000000,-0.828877,-0.028619,-0.512998
gear_ratio,-0.313661,0.616177,0.212842,-0.377852,-0.508665,-0.759258,-0.696383,-0.676300,-0.828877,1.000000,0.127787,0.521331


In [21]:
df_data.columns

Index(['make', 'price', 'mpg', 'rep78', 'headroom', 'trunk', 'weight',
       'length', 'turn', 'displacement', 'gear_ratio', 'foreign',
       'rep_missing', 'make_r', 'mpg_bins'],
      dtype='object')

In [75]:
def gen_results(equation,df_data,df_sum):
    model=smf.ols(equation, data=df_data).fit()
    r2=model.rsquared
    r2_adj=model.rsquared_adj
    aic=model.aic
    return pd.concat([df_sum,pd.DataFrame({'equation':[equation],'r2':[r2],'r2_adj':[r2_adj],'aic':[aic]})],axis=0)

In [76]:
df_sum=pd.DataFrame(columns=['equation','r2','r2_adj','aic'])

eq = 'mpg ~ weight'
df_sum=gen_results(eq,df_data,df_sum)

eq = 'mpg ~ weight + length'
df_sum=gen_results(eq,df_data,df_sum)

eq = 'mpg ~ weight + length + C(make_r)'
df_sum=gen_results(eq,df_data,df_sum)


In [77]:
df_sum

,equation,r2,r2_adj,aic
0,mpg ~ weight,0.651531,0.646691,394.777377
0,mpg ~ weight + length,0.661390,0.651852,394.653524
0,mpg ~ weight + length + C(make_r),0.787553,0.683497,404.158015


In [78]:
#eq = 'mpg ~ rep78:rep_missing + weight + displacement + C(foreign)'
df_data.columns

Index(['make', 'price', 'mpg', 'rep78', 'headroom', 'trunk', 'weight',
       'length', 'turn', 'displacement', 'gear_ratio', 'foreign',
       'rep_missing', 'make_r', 'mpg_bins'],
      dtype='object')